In [ ]:
!pip install timm -q

In [ ]:
import os, sys, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score

sys.path.append('/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/')

from src.config_2d import Config2D
from src.dataset_2d import SpectrogramDataset, preprocess_spectrograms
from src.model_2d import build_model_2d

In [ ]:
cfg = Config2D()
cfg.spectrogram_dir = '/kaggle/input/competitions/hms-harmful-brain-activity-classification/train_spectrograms/'
cfg.metadata_csv    = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_test_split.csv'
cfg.spec_cache_dir  = '/tmp/hms_spec_cache'

SEED = cfg.seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(cfg.as_dict())

In [ ]:
# Convert parquet → npy once; subsequent runs skip existing files.
preprocess_spectrograms(cfg.spectrogram_dir, cfg.spec_cache_dir, n_jobs=-1)

In [ ]:
train_ds = SpectrogramDataset(cfg.metadata_csv, cfg.spec_cache_dir, cfg, split='train')
val_ds   = SpectrogramDataset(cfg.metadata_csv, cfg.spec_cache_dir, cfg, split='val')

train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=cfg.num_workers, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                      num_workers=cfg.num_workers, pin_memory=True)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

# sanity-check batch shapes
batch = next(iter(train_dl))
print(f"image shape   : {batch['image'].shape}")
print(f"soft_label    : {batch['soft_label'].shape}")
print(f"label sample  : {batch['label'][:4]}")

In [ ]:
model = build_model_2d(cfg).to(cfg.device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone      : {cfg.backbone}")
print(f"Parameters    — total: {total_params:,} | trainable: {trainable_params:,}")

In [ ]:
# ========== SHARED EVALUATION SETUP ==========
# Same code path as the 1D CNN notebooks (src/evaluation.validate) so val
# numbers are computed the same way everywhere. It divides by
# len(loader.dataset) (not len(loader)), so an uneven last batch no longer
# biases val KL.
# NOTE: SpectrogramDataset's batch dict uses "image"/"label"/"soft_label"
# (not EEGDatasetV2's "x"/"y"/"soft_y"), so x_key/y_key/soft_key must be
# passed explicitly — soft_key="soft_y" would KeyError, there is no such
# key in this dataset.
from src.evaluation import validate
from src.losses import build_loss

eval_cfg = {"loss": {"name": "kl", "label_smoothing": 0.0},
            "model": {"num_classes": 6}}
# NOTE: must be the KLLoss from build_loss() — it applies log_softmax itself.
# Passing a bare nn.KLDivLoss() here would skip that and give wrong numbers.
val_loss_fn = build_loss(eval_cfg)

In [ ]:
DEVICE    = cfg.device
USE_AMP   = (DEVICE == 'cuda')
NUM_EPOCHS = cfg.num_epochs
patience   = cfg.patience

criterion = nn.KLDivLoss(reduction='batchmean')
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_kl, best_epoch, best_f1_at_best_kl, history = float('inf'), 0, 0.0, []
wait = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # ── train ────────────────────────────────────────────────────────────────
    model.train()
    t0         = time.time()
    train_loss = 0.0
    for batch in train_dl:
        images  = batch['image'].to(DEVICE)
        soft_y  = batch['soft_label'].to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(images)
            loss   = F.kl_div(F.log_softmax(logits, dim=1), soft_y, reduction='batchmean')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()

    # ── validate (shared: src/evaluation.validate) ──────────────────────────
    val_loss, metrics, val_probs, val_y_true = validate(
        model, val_dl, val_loss_fn, eval_cfg, DEVICE,
        soft_key='soft_label', x_key='image', y_key='label',
    )
    avg_val_kl = val_loss
    macro_f1   = metrics['macro_f1']
    avg_train  = train_loss / len(train_dl)
    elapsed    = time.time() - t0

    history.append({
        'epoch'     : epoch,
        'train_loss': avg_train,
        'val_kl'    : avg_val_kl,
        'macro_f1'  : macro_f1,
    })
    print(f"Epoch {epoch:03d} | train_kl {avg_train:.4f} | "
          f"val_kl {avg_val_kl:.4f} | "
          f"macro_f1 {macro_f1:.4f} | {elapsed:.0f}s")

    # checkpoint / early-stop on val_kl (the headline metric), not macro_f1
    if avg_val_kl < best_kl:
        best_kl    = avg_val_kl
        best_epoch = epoch
        best_f1_at_best_kl = macro_f1
        wait       = 0
        torch.save(model.state_dict(), 'best_cnn2d_v1.pt')
        print(f"  ✓ saved best_cnn2d_v1.pt (kl={best_kl:.4f})")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch} (patience={patience})")
            break

print(f"\nTraining complete. Best epoch {best_epoch}: "
      f"val_kl={best_kl:.4f}, macro_f1={best_f1_at_best_kl:.4f}")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
# best_epoch/best_kl/best_f1_at_best_kl are set by the checkpoint criterion in
# the training cell (best val_kl), so these two numbers are from the same model.
print(f"Best epoch    : {best_epoch}")
print(f"Best val KL   : {best_kl:.4f}")
print(f"Macro F1 there: {best_f1_at_best_kl:.4f}")

# ── Per-epoch log (copy-paste friendly for ablation table) ───────────────────
print(f"\n{'epoch':>5}  {'train_kl':>9}  {'val_kl':>7}  {'macro_f1':>9}")
for h in history:
    marker = ' ←' if h['epoch'] == best_epoch else ''
    print(f"{h['epoch']:>5}  {h['train_loss']:>9.4f}  "
          f"{h['val_kl']:>7.4f}  "
          f"{h['macro_f1']:>9.4f}{marker}")

# ── Plots ─────────────────────────────────────────────────────────────────────
epochs   = [h['epoch']     for h in history]
train_kl = [h['train_loss'] for h in history]
val_kl   = [h['val_kl']    for h in history]
macro_f1 = [h['macro_f1']  for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(epochs, train_kl, label='train KL')
ax1.plot(epochs, val_kl,   label='val KL')
ax1.axvline(best_epoch, color='red', linestyle='--', label=f'best epoch={best_epoch}')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('KL Divergence')
ax1.set_title('KL Divergence'); ax1.legend()

ax2.plot(epochs, macro_f1, color='green')
ax2.axvline(best_epoch, color='red', linestyle='--', label=f'macro_f1={best_f1_at_best_kl:.3f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
ax2.set_title('Validation Macro F1'); ax2.legend()

plt.tight_layout()
plt.savefig('cnn2d_v1_curves.png', dpi=150)
plt.show()

# One-line summary — copy into notebooks/RESULTS.md
print(f"\ncnn-2d-v1-baseline | "
      f"val_kl={best_kl:.4f} | "
      f"val_f1={best_f1_at_best_kl:.4f} | "
      f"epoch={best_epoch}")